# Attention Matrislerinden Rule-Based Subject/Object Tespiti
Bu projede, `xlm-roberta-base` modelinin attention matrisleri kullanılarak subject/object tespiti yapılmaktadır. Tokenizer'ın yapısı nedeniyle her kelime birden fazla token'a bölünebilir. Bu nedenle label'lar da genişletilmektedir.

In [1]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
import numpy as np

In [3]:
df = pd.read_excel('/content/dataset.xlsx')
df['tokens'] = df['tokens'].apply(eval)
df['labels'] = df['labels'].apply(eval)

In [4]:
model_name = 'xlm-roberta-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name, output_attentions=True)
model.eval()

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

XLMRobertaModel(
  (embeddings): XLMRobertaEmbeddings(
    (word_embeddings): Embedding(250002, 768, padding_idx=1)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): XLMRobertaEncoder(
    (layer): ModuleList(
      (0-11): 12 x XLMRobertaLayer(
        (attention): XLMRobertaAttention(
          (self): XLMRobertaSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): XLMRobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine

In [5]:
def expand_labels(tokens, original_tokens, original_labels):
    expanded_labels = []
    token_pointer = 0
    for word, label in zip(original_tokens, original_labels):
        reconstructed = ''
        label_tokens = []
        while token_pointer < len(tokens):
            token = tokens[token_pointer].lstrip('▁')
            reconstructed += token
            label_tokens.append(label)
            token_pointer += 1
            if reconstructed == word.replace('’', "'"):
                break
        expanded_labels.extend(label_tokens)
    return expanded_labels

In [6]:
def get_attention(sentence):
    inputs = tokenizer(sentence, return_tensors='pt')
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.attentions, tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

def rule_based_prediction(attentions, verb_index):
    avg_attention = torch.stack(attentions).mean(dim=0)[0].mean(dim=0)
    subj_scores = avg_attention[:, verb_index]
    obj_scores = avg_attention[verb_index, :]
    return subj_scores.argmax().item(), obj_scores.argmax().item()

In [7]:
correct_subj = correct_obj = total = 0

for _, row in df.iterrows():
    sentence, tokens_, labels_ = row['sentence'], row['tokens'], row['labels']
    if 'verb' not in labels_:
        continue
    verb_idx = labels_.index('verb')
    attentions, tokens = get_attention(sentence)
    expanded_labels = expand_labels(tokens[1:-1], tokens_, labels_)  # [CLS] ve [SEP] hariç
    pred_subj, pred_obj = rule_based_prediction(attentions, verb_idx + 1)  # +1 çünkü [CLS] başta
    try:
        gold_subj = expanded_labels.index('subject')
        gold_obj = expanded_labels.index('object')
    except ValueError:
        continue
    correct_subj += (pred_subj == gold_subj)
    correct_obj += (pred_obj == gold_obj)
    total += 1

print(f'Subject Accuracy: {correct_subj / total:.2f}')
print(f'Object Accuracy: {correct_obj / total:.2f}')

XLMRobertaSdpaSelfAttention is used but `torch.nn.functional.scaled_dot_product_attention` does not support non-absolute `position_embedding_type` or `output_attentions=True` or `head_mask`. Falling back to the manual attention implementation, but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.


Subject Accuracy: 0.01
Object Accuracy: 0.15
